In [1]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [10]:
import streamlit as st
import geopy
from geopy.distance import geodesic
import json
import random
import geopandas as gpd

In [11]:
# 計算兩點距離（公里）
def haversine(lon1, lat1, lon2, lat2):
    return geodesic((lat1, lon1), (lat2, lon2)).kilometers

In [12]:
# 載入餐廳資料
@st.cache_data
def load_restaurant_data(filename="restaurant-tpe.geojson"):
    with open(filename, encoding="utf-8") as f:
        return json.load(f)

2025-05-12 11:09:44.276 No runtime found, using MemoryCacheStorageManager


In [13]:
# 查找附近餐廳
def find_nearby_restaurants(target_lat, target_lon, data, radius_km=1.0):
    nearby = []
    for feature in data["features"]:
        if feature["geometry"]["type"] != "Point":
            continue
        lon, lat = feature["geometry"]["coordinates"]
        distance = haversine(target_lon, target_lat, lon, lat)
        if distance <= radius_km:
            nearby.append(feature)
    return nearby


In [14]:
# 推薦一間餐廳
def recommend_random_restaurant(nearby):
    if nearby:
        selected = random.choice(nearby)
        name = selected["properties"].get("name", "未命名餐廳")
        lon, lat = selected["geometry"]["coordinates"]
        return name, lat, lon
    return None, None, None

In [15]:
# 主程式
def main():
    st.title("台大附近餐廳推薦")
    st.write("輸入台大一棟建築物，系統將推薦附近一公里內的隨機餐廳。")

    # 載入建築物資料
    buildings = gpd.read_file("ntu_buildings.geojson")

    # 使用者輸入建築物名稱
    building_name = st.text_input("請輸入建築物名稱（例如：綜合教學館）")

    if building_name:
        matched = buildings[buildings["name"].str.contains(building_name, na=False)]
        if matched.empty:
            st.error("找不到這棟建築物，請重新輸入。")
            return

        # 建築物中心點
        geom = matched.iloc[0].geometry
        point = geom.centroid if geom.geom_type == "Polygon" else geom
        target_lat, target_lon = point.y, point.x

        # 載入餐廳
        data = load_restaurant_data()

        # 查詢並推薦
        nearby = find_nearby_restaurants(target_lat, target_lon, data)
        name, lat, lon = recommend_random_restaurant(nearby)

        if name:
            st.success(f"推薦餐廳：{name}")
            st.map(data=[{"lat": lat, "lon": lon}], zoom=15)
        else:
            st.warning("附近沒有找到餐廳。")

In [16]:
if __name__ == "__main__":
    main()

2025-05-12 11:09:55.501 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:09:55.503 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:09:55.504 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:09:55.505 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:09:55.507 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:09:55.511 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
Skipping field ways: unsupported OGR type: 1
2025-05-12 11:09:56.262 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:09:56.264 Thread 'MainThread': missing ScriptRunContext! Th

In [17]:
# Streamlit 介面
st.title("台大建築物附近餐廳推薦")

# 輸入
building_name = st.text_input("請輸入台大建築物名稱（例如：綜合教學館）")

if building_name:
    buildings = load_buildings()
    matched = buildings[buildings['name'].str.contains(building_name, na=False)]

    if matched.empty:
        st.warning("找不到這棟建築物")
    else:
        geom = matched.iloc[0].geometry
        centroid = geom.centroid if geom.geom_type == "Polygon" else geom
        lat, lon = centroid.y, centroid.x

        restaurant_data = load_restaurant_data()
        nearby = find_nearby_restaurants(lat, lon, restaurant_data)
        result = recommend_random_restaurant(nearby)

        st.subheader("推薦結果")
        st.write(result)

2025-05-12 11:10:05.611 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:10:05.614 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:10:05.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:10:05.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:10:05.616 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:10:05.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:10:05.618 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-12 11:10:05.618 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [9]:
streamlit run app.py

SyntaxError: invalid syntax (<ipython-input-9-718866ff34b9>, line 1)